In [1]:
import warnings
warnings.filterwarnings('ignore')
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from llama_index.core.agent.workflow import FunctionAgent, AgentWorkflow
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
llm = Groq(model="qwen/qwen3.6-27b", max_tokens=4096)
Settings.llm = llm

In [3]:
documents = SimpleDirectoryReader(input_files=["pairs.pdf"]).load_data()
index = VectorStoreIndex.from_documents(documents, show_progress=False)
query_engine = index.as_query_engine()
print(f"Index created with {len(documents)} document chunks")

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.77it/s]

Index created with 10 document chunks


In [4]:
async def query_pdf(question: str) -> str:
    """Query the pairs trading PDF for specific information."""
    response = query_engine.query(question)
    return str(response)


async def extract_strategy() -> str:
    """Extract the complete trading strategy description from the PDF."""
    query = (
        "Extract a detailed description of the pairs trading strategy from this document. "
        "Include: 1) The assets being traded, 2) The statistical methods used, "
        "3) The entry and exit signals, 4) Any risk management rules."
    )
    response = query_engine.query(query)
    return str(response)

In [5]:
strategy_agent = FunctionAgent(
    name="TradingStrategyAgent",
    description="Converts trading strategy PDFs into Python code.",
    system_prompt="""You are a trading strategy implementation specialist. Your task is to:

1. First, use extract_strategy to get the trading strategy from the PDF
2. Then, use query_pdf if you need additional details about specific aspects
3. Finally, create a complete implementation plan AND Python code

When you have gathered enough information, provide your response in this exact format:

IMPLEMENTATION PLAN:
[Your detailed plan here]

PYTHON CODE:
```python
[Your complete, executable Python code here]
```

The Python code should be complete and runnable, including all imports, functions, 
and a main execution block. Focus on the core strategy logic - use pandas, numpy, 
and statsmodels for the implementation.
""",
    llm=llm,
    tools=[extract_strategy, query_pdf],
)


In [6]:
agent_workflow = AgentWorkflow(
    agents=[strategy_agent],
    root_agent=strategy_agent.name,
)

In [7]:
handler = agent_workflow.run(
    user_msg=(
        "Read the pairs.pdf file and convert the trading strategy into Python code. "
        "First extract the strategy, then create an implementation plan and generate "
        "complete, executable Python code."
    )
)

response = await handler
final_response = response.response.content

2026-07-22 17:59:14,227 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 22.57it/s]
2026-07-22 17:59:15,053 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 48.41it/s]
2026-07-22 17:59:15,902 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 76.02it/s]
2026-07-22 17:59:17,502 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [8]:
if "IMPLEMENTATION PLAN:" in final_response:
    parts = final_response.split("PYTHON CODE:")
    plan = parts[0].replace("IMPLEMENTATION PLAN:", "").strip()
    
    if len(parts) > 1:
        code = parts[1].strip()
        if "```python" in code:
            code = code.split("```python")[1].split("```")[0].strip()
        elif "```" in code:
            code = code.split("```")[1].split("```")[0].strip()
else:
    plan = ""
    code = final_response

In [9]:
print("=" * 50)
print("GENERATED PYTHON CODE")
print("=" * 50)
print(code[:3000] + "..." if len(code) > 3000 else code)

GENERATED PYTHON CODE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

class PairsTrader:
    def __init__(self, entry_threshold=2.0, exit_threshold=0.5, lookback_period=60):
        """
        Initialize the pairs trader
        
        Parameters:
        -----------
        entry_threshold : float
            Z-score threshold for entering trades
        exit_threshold : float
            Z-score threshold for exiting trades
        lookback_period : int
            Number of periods for rolling calculations
        """
        self.entry_threshold = entry_threshold
        self.exit_threshold = exit_threshold
        self.lookback_period = lookback_period
        self.positions = []
        self.trades = []
        
    def test_cointegration(self, price1,